In [1]:
import os
import time
import requests
import pandas as pd

# ListenBrainz connection
LB_BASE_URL = "https://api.listenbrainz.org/1"
LB_TOKEN = os.getenv("LISTENBRAINZ_TOKEN")  # optional; raises rate limits if set

lb = requests.Session()
lb.headers.update({"User-Agent": "TeamTalentScouts/0.1 (MADS Capstone)"})
if LB_TOKEN:
    lb.headers.update({"Authorization": f"Token {LB_TOKEN}"})


def lb_get(endpoint, params=None, max_retries=3):
    """GET a ListenBrainz endpoint, backing off on rate limits (429)."""
    url = f"{LB_BASE_URL}/{endpoint.lstrip('/')}"
    for _ in range(max_retries):
        resp = lb.get(url, params=params, timeout=30)
        if resp.status_code == 429:
            time.sleep(int(resp.headers.get("X-RateLimit-Reset-In", 5)))
            continue
        resp.raise_for_status()
        return resp
    resp.raise_for_status()

In [2]:
# Test the connection: pull the top 10 sitewide artists (all time)
start = time.time()
resp = lb_get("stats/sitewide/artists", params={"range": "all_time", "count": 10})
elapsed = time.time() - start

print(f"Status: {resp.status_code} | Latency: {elapsed:.2f}s")
print(f"Rate limit remaining: {resp.headers.get('X-RateLimit-Remaining')} / {resp.headers.get('X-RateLimit-Limit')}")

payload = resp.json()["payload"]
top_artists = pd.DataFrame(payload["artists"])
top_artists

Status: 200 | Latency: 0.76s
Rate limit remaining: 29 / 30


,artist_mbid,artist_name,listen_count
0,a74b1b7f-71a5-4011-9441-d0b5e4122711,Radiohead,3624185
1,f59c5520-5f46-4d2c-b2c4-822eabf53419,Linkin Park,3082594
2,056e4f3e-d505-4dad-8ec1-d04f521cbb56,Daft Punk,2969751
3,b10bbbfc-cf9e-42e0-be17-e2c3e1d2600d,The Beatles,2944018
4,381086ea-f511-4aba-bdf9-71c753dc5077,Kendrick Lamar,2853811
5,164f0d73-1234-4e2c-8743-d77bf2191051,Ye,2832351
6,83d91898-7763-47d7-b03b-b92132375c47,Pink Floyd,2629162
7,e21857d5-3256-4547-afb3-4b6ded592596,Gorillaz,2511286
8,cc197bad-dc9c-440d-a5b5-d52ba2e14234,Coldplay,2409583
9,b95ce3ff-3d05-4e87-9e01-c97b66af13d4,Eminem,2406625


In [ ]:
ranges = ("all_time", "year", "this_year")
artist_results = {}

for artist_range in ranges:
    response = lb_get(
        "stats/sitewide/artists",
        params={"range": artist_range, "count": 100},
    )
    artists = response.json()["payload"]["artists"]
    artist_results[artist_range] = pd.DataFrame(artists)
    artist_results[artist_range]["range"] = artist_range
    artist_results[artist_range]["rank"] = range(1, len(artists) + 1)

artist_baseline = artist_results["all_time"].copy()

recent_artists = pd.concat(
    [artist_results["year"], artist_results["this_year"]],
    ignore_index=True,
)
recent_artists = recent_artists.drop_duplicates(subset="artist_mbid")

baseline_mbids = set(artist_baseline["artist_mbid"].dropna())
candidate_seeds = recent_artists[
    ~recent_artists["artist_mbid"].isin(baseline_mbids)
].copy()

artist_baseline.head(), candidate_seeds.head()

Payload keys: ['artists', 'count', 'from_ts', 'last_updated', 'offset', 'range', 'to_ts', 'total_artist_count']
Artist record keys: ['artist_mbid', 'artist_name', 'listen_count']
